In [ ]:
#Importing all the necessary libraries and modules
import numpy as np
import pandas as pd
from scipy.stats import chi2
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import statsmodels.graphics.tsaplots as sgt
import statsmodels.tsa.stattools as sts
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, root_mean_squared_error , r2_score , mean_squared_error
from pmdarima.arima import auto_arima
from pmdarima.arima import OCSBTest 
from arch import arch_model
import seaborn as sns
import yfinance
import warnings
warnings.filterwarnings("ignore")
sns.set()

### We will work on the NIFTY 50 Stock Index

In [ ]:
# Importing the raw NIFTY 50 data using yfinance 
raw_nifty_data = yfinance.download (tickers = "^NSEI", #The time series we are interested in - (in our case, this is NIFTY 50 )
                              start = "2007-09-17", #The starting date of our data set
                              end = "2026-04-30", #The ending date of our data set (at the time of upload, this is the current date)
                              interval = "1d", #The distance in time between two recorded observations. Since we're using daily closing prices, we set it equal to "1d", which indicates 1 day. 
                              group_by = 'ticker', #The way we want to group the scraped data. Usually we want it to be "ticker", so that we have all the information about a time series in 1 variable.
                              auto_adjust = True, #Automatically adjuss the closing prices for each period. 
                              threads = True) #Whether to use threads for mass downloading. 

In [ ]:
raw_nifty_data.info()

In [ ]:
raw_nifty_data.head()

`Saving Our Raw Data into csv for later use`

In [ ]:
# raw_nifty_data.to_csv("OHLCV_nifty.csv")

In [ ]:
# Making a copy of the raw data for all the manipulation
nifty_df = raw_nifty_data.copy()

In [ ]:
print(nifty_df.shape)
print(nifty_df.describe())

In [ ]:
nifty_df.columns = nifty_df.columns.droplevel(0)

In [ ]:
nifty_df.head()

In [ ]:
# Data Cleaning and Preprocessing
nifty_df['price'] = nifty_df['Close']

In [ ]:
# Delete other columns which are not necessary
del nifty_df['Open']
del nifty_df['Close']
del nifty_df['High']
del nifty_df['Low']
del nifty_df['Volume']

In [ ]:
# Now we only have closing price with column name as price
nifty_df.head()

In [ ]:
nifty_df = nifty_df.asfreq("B") # business days
nifty_df = nifty_df.ffill() # forward fill to remove the NaN prices

In [ ]:
nifty_df.info()

In [ ]:
# Forming Log Returns 
nifty_df['returns'] = np.log(nifty_df['price']/ nifty_df['price'].shift(1))

`Cool We are done with our CLeaning and Pre Processing`

In [ ]:
nifty_df.to_pickle("nifty_index.pkl") # Saving the final cleaned data in pickle format to save the datatype and processing preserved.

In [ ]:
# Let us Train- test split the data with 80-20 form
mark = int(len(nifty_df)*0.8)
nifty_train = nifty_df.iloc[:mark]
nifty_test = nifty_df.iloc[mark:]

In [ ]:
print(f'Shape of training data -> {nifty_train.shape}')
print(f'Shape of testing data -> {nifty_test.shape}')

#### Plotting The Data 

In [ ]:

plt.figure(figsize=(14, 5))
plt.plot(nifty_df['price'], color='steelblue', linewidth=1)
plt.title('NIFTY 50 Price (2010 - 2026)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (INR)')
plt.tight_layout()
plt.show()

In [ ]:
# Summary Statistics
print(nifty_df.describe())

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(nifty_df['price'], color='steelblue', linewidth=1, label='NIFTY 50 Price')
plt.plot(nifty_df['price'].rolling(252).mean(), color='red', linewidth=1.5, label='1 Year Rolling Mean')
plt.title('NIFTY 50 Price (2010 - 2026)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (INR)')
plt.legend()
plt.tight_layout()
plt.show()

#### ***The rolling mean keeps shifting upward over time — which is a classic visual sign of non-stationarity. A stationary series would have a flat, constant rolling mean.***

In [ ]:
# Plotting ACF and PACF On Closing Prices
sgt.plot_acf(nifty_df.price , zero=False,lags=40)
plt.show()

In [ ]:
sgt.plot_pacf(nifty_df.price , zero=False,lags=40, alpha=0.05, method=('ols'))
plt.ylim(-.1,.1)
plt.show()

`FROM ACF, we can see that the Prices are Non- Stationary`                                                                 
`To confirm this -> We run Augmented Dickey Fuller Test`

In [ ]:
t_stats, p_value , *other  = sts.adfuller(nifty_df.price)

In [ ]:
print(f'T-Statistic = {t_stats} \n P-Value = {p_value}')

`Since p value > 0.05 Thus we cannot reject the Null Hyppothesis -> Closing Prices is Non Stationary`

#### ***This is the plot of training and testing data splitted***

In [ ]:
plt.figure(figsize=(14, 5))

# Plot train data
plt.plot(nifty_train.index, nifty_train['price'], color='steelblue', linewidth=1, label='Train Data')

# Plot test data
plt.plot(nifty_test.index, nifty_test['price'], color='darkorange', linewidth=1, label='Test Data')

# Shaded region for test data
plt.axvspan(nifty_test.index[0], nifty_test.index[-1], alpha=0.2, color='darkorange', label='Test Region')

# Split line
plt.axvline(nifty_test.index[0], color='red', linewidth=1.2, linestyle='--', label='Train/Test Split')

plt.title('NIFTY 50 — Train / Test Split (80/20)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (INR)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plotting the Daily log Returns
plt.figure(figsize=(20,5))
plt.plot(nifty_df.returns[1:],color='g',label='Returns')
plt.axhline(nifty_df.returns.mean(),color='k',linestyle='--',label='mean line')
plt.title("Daily Log Returns NIFTY50",size=24)
plt.legend()
plt.show()

In [ ]:
# Histogram 
plt.hist(nifty_df.returns,bins=75,label='Histogram')
plt.axvline(nifty_df.returns.mean(),color='r',linestyle='--',label='Mean Line')
plt.title("Histogram Log Returns",size=16)
plt.legend()
plt.show()


`Clearly this shows that Log Returns of NIFTY50 is Normally distributed`

In [ ]:
# Checking the Stationarity of Returns
sgt.plot_acf(nifty_df.returns[1:] , zero=False, lags=40)
plt.ylim(-.1,.1)
plt.show()

In [ ]:
# Since most of the lags are in significant -> Stationarity 
# Let us Confirm it by ADF Test
t_stats_ret, p_value_ret , *other  = sts.adfuller(nifty_df.returns[1:])

In [ ]:
print(f'\tADF TEST ON LOG RETURNS \nT-Statistic = {t_stats_ret} \n P-Value = {p_value_ret}')

`We clearly see that p value is <0.05 Thus We can rejec the Null Hypothesis and Thus Log Returns are Stationary`

### Model Training

"Before applying advanced time series models like AR and MA, we first establish baseline models. Baseline models are simple forecasting methods that require minimal assumptions and computation.

They serve as a benchmark — any advanced model we build must perform better than these baselines to be considered useful. If our AR or MA model cannot beat a simple baseline, it means the model is not adding any value.

We apply the following 3 baseline models on the NIFTY 50 closing price:

Naive Model — assumes tomorrow's price equals today's price

Simple Average Model — assumes tomorrow's price equals the average of all past prices

Moving Average Model — assumes tomorrow's price equals the average of the last n days

After forecasting, we evaluate each model using error metrics (MAE, RMSE, MAPE) and compare their performance."

#### Baseline Models

`(i) Mean Forecast`

In [ ]:
# Simple Average Model
# Forecast = mean of all train prices
nifty_test['avg_forecast'] = nifty_train['price'].mean()
nifty_test.head()

In [ ]:
# Plot
plt.figure(figsize=(14,5))
plt.plot(nifty_train['price'], color='steelblue', label='Train')
plt.plot(nifty_test['price'], color='green', label='Actual')
plt.plot(nifty_test['avg_forecast'], color='purple', linestyle='--', label='Average Forecast')
plt.title('Simple Average Model Forecast', fontweight='bold')
plt.legend()
plt.show()

In [ ]:
# Mean residuals
mean_residual = nifty_test['price'] - nifty_test['avg_forecast']

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

# residual line
plt.plot(mean_residual, color='red', label='Residuals')

# zero reference line
plt.axhline(y=0, color='black', linestyle='--')

# titles
plt.title("Residuals (Actual – Forecast)")

# labels
plt.xlabel("Date")
plt.ylabel("Residuals")

plt.legend()
plt.show()

`Evaluating MAE, RMSE`

In [ ]:
mae = np.mean(abs(nifty_test.price - nifty_test.avg_forecast))
rmse = np.sqrt(np.mean((nifty_test.price - nifty_test.avg_forecast)**2))
mape = np.mean(np.abs((nifty_test['price'] - nifty_test['avg_forecast']) / nifty_test['price'])) * 100

print('=== Simple Average Model Errors ===')
print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAPE : {mape:.2f}%')

`Thus We see that Mean Absolute Error and Root Mean Squared Error are very high Thus suggesting the very bad fitting`

"We evaluate our models using three error metrics:

MAE tells us the average error in rupees — how far off our forecast is on a typical day.

RMSE tells us if our model is making large errors on some days — it penalizes big mistakes more than small ones.

MAPE tells us the error as a percentage — making it easy to compare across all models regardless of the price level."

In [ ]:
# STORING RESULT FOR LATER COMPARISION
results = pd.DataFrame(columns=['Model', 'MAE', 'RMSE', 'MAPE'])
results.loc[0] = ['Mean', mae, rmse, mape]
print(results)

`(ii) Naive Model`

`One of the best performing Baseline model, it assumes that the forecast value is nothing but the last recorded data`       
`If we fit the Naive Model on Training data then it assumes a fix value which is the last record of training data`

In [ ]:
# Last value of train data
naive_forecast = nifty_test['price'].copy()
naive_forecast[:] = nifty_train['price'].iloc[-1]
nifty_test['naive_forecast'] = nifty_train['price'].iloc[-1]
nifty_test.head()

#### Plotting Naive Model

In [ ]:
plt.figure(figsize=(20,10))
plt.plot(nifty_train.price,color='b',label='Train Data')
plt.plot(nifty_test.price,color='orange',label='Test Data')
plt.axvline(nifty_train.index[-1],linestyle="--",color='k')
plt.plot(nifty_test.naive_forecast, color='green',linestyle='--',label='Naive Forecast')
plt.title("NIFTY50 Mean Forecast",size=24)
plt.xlabel("Year")
plt.ylabel("Closing Prices")
plt.legend()
plt.show()

In [ ]:
# Residual Series of Naive Model
naive_residual = nifty_test.price - nifty_test.naive_forecast

In [ ]:
# Plotting Residuals of Naive Model

plt.figure(figsize=(12,6))

# residual line
plt.plot(naive_residual, color='red', label='Residuals')

# zero reference line
plt.axhline(y=0, color='black', linestyle='--')

# titles
plt.title("Residuals (Actual – Forecast)")

# labels
plt.xlabel("Date")
plt.ylabel("Residuals")

plt.legend()
plt.show()

In [ ]:
# MAE
mae = mean_absolute_error(nifty_test['price'], nifty_test['naive_forecast'])

# RMSE
rmse = root_mean_squared_error(nifty_test['price'], nifty_test['naive_forecast'])

# MAPE
mape = np.mean(np.abs((nifty_test['price'] - nifty_test['naive_forecast']) / nifty_test['price'])) * 100

print('=== Naive Model Errors ===')
print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAPE : {mape:.2f}%')

In [ ]:
# Update the Dataframe with naive errors results
results.loc[1] = ['Naive', mae, rmse, mape]
results = results.sort_values(by='MAPE')
print(results.head())

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(nifty_train['price'].iloc[-100:], 
         color='steelblue', label='Train', linewidth=1)
plt.plot(nifty_test['price'], 
         color='black', linewidth=2, label='Actual')
plt.plot(nifty_test['naive_forecast'], 
         '--', color='red', linewidth=1, label='Naive')
plt.plot(nifty_test['avg_forecast'],   
         '--', color='purple', linewidth=1, label='Simple Avg')
plt.axvline(nifty_test.index[0], color='black', 
            linestyle=':', linewidth=1)
plt.title('All Models — Forecast vs Actual NIFTY', 
          fontsize=14, fontweight='bold')
plt.ylabel('Price (INR)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    ax.bar(results['Model'], results[metric], 
           color=['steelblue','purple','darkorange','red','green'],
           edgecolor='white')
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('All Models — Error Metric Comparison', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

#### ***-------3.)FORCASTING LAST 30 DAYS MEAN --------***

In [ ]:
# Moving Average Model
# Forecast = average of last 30 days of train data
window = 30
nifty_test['30 days mean'] = nifty_train['price'].rolling(window=window).mean().iloc[-1]
nifty_test.head()

#### Now let us plot our data and visualise

In [ ]:
# Plot
plt.figure(figsize=(14,5))
plt.plot(nifty_train['price'], color='steelblue', label='Train')
plt.plot(nifty_test['price'], color='green', label='Actual')
plt.plot(nifty_test['30 days mean'], color='darkorange', linestyle='--', label=f'Moving Average Forecast (window={window})')
plt.title('Moving Average Model Forecast', fontweight='bold')
plt.legend()
plt.show()

In [ ]:
# Error Metrics
mae  = mean_absolute_error(nifty_test['price'], nifty_test['30 days mean'])
rmse = np.sqrt(mean_squared_error(nifty_test['price'], nifty_test['30 days mean']))
mape = np.mean(np.abs((nifty_test['price'] - nifty_test['30 days mean']) / nifty_test['price'])) * 100
print('=== Moving Average Model Errors ===')
print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAPE : {mape:.2f}%')

In [ ]:
# Add to results
results.loc[2] = ['30 days mean', mae, rmse, mape]
results = results.sort_values(by='MAPE')
print(results.head())

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(nifty_train['price'].iloc[-100:], 
         color='steelblue', label='Train', linewidth=1)
plt.plot(nifty_test['price'], 
         color='black', linewidth=2, label='Actual')
plt.plot(nifty_test['naive_forecast'], 
         '--', color='red', linewidth=1, label='Naive')
plt.plot(nifty_test['avg_forecast'],   
         '--', color='purple', linewidth=1, label='Simple Avg')
plt.plot(nifty_test['30 days mean'],   
         '--', color='darkorange', linewidth=1, label='30 Days Mean')
plt.axvline(nifty_test.index[0], color='black', 
            linestyle=':', linewidth=1)
plt.title('All Models — Forecast vs Actual NIFTY', 
          fontsize=14, fontweight='bold')
plt.ylabel('Price (INR)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

### ***BAR GRAPH***

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    ax.bar(results['Model'], results[metric], 
           color=['steelblue','purple','darkorange','red','green'],
           edgecolor='white')
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('All Models — Error Metric Comparison', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

`We see that in baseline models -> Naive has the least error`

### ***ADVANCED MODELS***

## ***-------4.)AR model --------***

"After establishing our baseline models, we now 
apply the AR (AutoRegressive) model.

An AR model predicts future values using a 
linear combination of its own past values.
The equation is:

y(t) = c + φ1*y(t-1) + φ2*y(t-2) + ... + φp*y(t-p) + ε

Where:
y(t)      = current value we want to predict
c         = constant
φ         = coefficients (weights given to past values)
y(t-1)    = value from 1 period ago
p         = number of past values used (order)
ε         = error term

The key parameter is p (order) which tells the 
model how many past values to look at.

From our PACF plot we observed significant spikes 
at lag 5 and lag 6, therefore we will use p = 6 
for our AR model.

Important: AR model requires stationary data, 
which is why we fit it on log returns instead 
of raw prices. After forecasting log returns 
we convert them back to prices for comparison."

#### **First we have to find the order of the AR model then we can proceed for that we have to look at the PACF graph again**

### ***Why do we use PACF for AR order and ACF for MA order?***

"A common question is — why do we look at the 
PACF graph to find the order of AR model and 
not the ACF graph?

The answer lies in what each model uses for 
prediction and what each graph measures.

The AR model predicts future values using past 
actual values of the series. We need to find 
the direct relationship between current value 
and each past value. PACF measures exactly this 
— it shows the pure direct correlation between 
current and past values by removing the effect 
of all intermediate lags. So PACF tells us 
exactly at which lag the direct effect stops, 
and that becomes our AR order (p).

If we used ACF for AR order, we would get a 
wrong answer because ACF includes both direct 
and indirect effects mixed together, which would 
overestimate the number of lags needed.

Similarly the MA model predicts future values 
using past errors. ACF measures the total 
correlation including all indirect effects, 
which matches exactly how MA model works. 
So ACF is used to find MA order (q).

In simple terms:
PACF → direct effect only → AR model
ACF  → total effect       → MA model"

In [ ]:
nifty_train.head()

In [ ]:
# Let us plot PACF for log returns 
sgt.plot_pacf(nifty_train['returns'][1:] ,
                lags=40 ,
                method='ols',
                zero=True)
plt.ylim(-.25,.25)
plt.show()

#### ***"From the PACF plot we can observe a very slight spike at lag 4 and then significant spike at lag 7, which initially suggests that AR(4) or AR(7) could be the appropriate order for our AR model.***

However we cannot rely on the PACF plot alone 
to select the final order because:

1. PACF can sometimes show false significant 
   spikes due to randomness in the data

2. We need statistical confirmation that adding 
   each lag is actually improving the model 
   significantly

Therefore we will formally determine the best 
AR order by:

1. Fitting AR models from AR(1) to AR(7) 
   one by one

2. Checking AIC and BIC at each step
   → Lower AIC/BIC = better model

3. Applying the LLR (Log Likelihood Ratio) test 
   at each step to check if adding a new lag 
   significantly improves the model
   → p-value < 0.05 = significant improvement
   → p-value > 0.05 = no significant improvement

The model where LLR test becomes insignificant 
(p > 0.05) will be our final AR order."

In [ ]:
## LLR test
def LLR_test(mod_1, mod_2, DF=1):
    L1 = mod_1.fit().llf
    L2 = mod_2.fit().llf
    LR = (2*(L2-L1))
    p = chi2.sf(LR, DF).round(3)
    return p

In [ ]:
log_ret = nifty_train['returns'].dropna()

# AR(1)
model_ar1 = ARIMA(log_ret, order=(1,0,0))
ar1_fitted = model_ar1.fit()
print('=== AR(1) ===')
print(f'AIC : {ar1_fitted.aic:.4f}')
print(f'BIC : {ar1_fitted.bic:.4f}')
print(ar1_fitted.summary())

In [ ]:
# Fitting AR(2)

model_ar2 = ARIMA(log_ret, order=(2,0,0))
ar2_fitted = model_ar2.fit()
print('=== AR(2) ===')
print(f'AIC : {ar2_fitted.aic:.4f}')
print(f'BIC : {ar2_fitted.bic:.4f}')
print(ar2_fitted.summary())

##### ***We clearly see that AIC/BIC instead of decreasing it got increased. Thus AR(2) Model is actually worse than AR(1)***

In [ ]:
# Applying LLR test and comparing AR(1) and AR(2)
print(LLR_test(model_ar1,model_ar2))

`Since we saw a spike at lag 4 and 7 so just to be on the safe side let us fit those also`

In [ ]:
# Fitting AR(4)

model_ar4 = ARIMA(log_ret, order=(4,0,0))
ar4_fitted = model_ar4.fit()
print('=== AR(4) ===')
print(f'AIC : {ar4_fitted.aic:.4f}')
print(f'BIC : {ar4_fitted.bic:.4f}')
print(ar4_fitted.summary())

In [ ]:
# Applying LLR test and comparing AR(1) and AR(4)
print(LLR_test(model_ar1,model_ar4 , DF=3))

In [ ]:
# Fitting AR(7)

model_ar7 = ARIMA(log_ret, order=(7,0,0))
ar7_fitted = model_ar7.fit()
print('=== AR(7) ===')
print(f'AIC : {ar7_fitted.aic:.4f}')
print(f'BIC : {ar7_fitted.bic:.4f}')
print(ar7_fitted.summary())

In [ ]:
# Applying LLR test and comparing AR(1) and AR(7)
print(LLR_test(model_ar1,model_ar7 , DF=6))

`Now this is interesting since we got p value of LLR Test = 0 after degree of Freedom = 6, it is very significant`

**Conclusion**

`Model fit (AIC/BIC)`                                        
***AR(7) has a lower AIC, which means it fits the data better *after penalizing for the extra 6 parameters. However, BIC tells a slightly different story — AR(7)'s BIC is considerably higher than AR(1)'s. BIC penalizes complexity more aggressively than AIC. This is a classic tension: AIC rewards predictive fit, BIC rewards parsimony. For financial returns where overfitting is a real concern, BIC leaning toward AR(1) is not a signal to ignore.***                                                 

`Log-Likelihood Ratio (LLR) Test — p = 0`
***The LLR test asks: does adding lags 2–7 significantly improve fit over AR(1)? With p ≈ 0, the answer is yes — the improvement is statistically significant, not random. This is the strongest argument for AR(7). The log-likelihood went from 11194.84 to 11212.36, a difference of ~17.5. For a chi-squared test with 6 degrees of freedom, that's overwhelmingly significant.***

`Now Next step is to calculate the Residuals and The Errors after forecasting and then compare and choose the better model between AR(1) and AR(7)`

In [ ]:
# Forecast log returns for test period with AR(1) Model
forecast_log_ret_ar1 = ar1_fitted.forecast(steps=len(nifty_test))

# Convert log returns back to prices
last_train_price = nifty_train['price'].iloc[-1]
ar1_prices = []

for lr in forecast_log_ret_ar1:
    last_train_price = last_train_price * np.exp(lr)
    ar1_prices.append(last_train_price)

# Store in test dataframe
nifty_test['ar1_forecast'] = ar1_prices

In [ ]:
nifty_test.tail()

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(nifty_train['price'], color='steelblue', label='Train')
plt.plot(nifty_test['price'], color='green', label='Actual')
plt.plot(nifty_test['ar1_forecast'], color='red', 
         linestyle='--', label='AR(1) Forecast')
plt.title('AR(1) Model Forecast vs Actual NIFTY', fontweight='bold')
plt.legend()
plt.show()

In [ ]:
# Forecast log returns for test period with AR(7) Model
forecast_log_ret_ar7 = ar7_fitted.forecast(steps=len(nifty_test))

# Convert log returns back to prices
last_train_price = nifty_train['price'].iloc[-1]
ar7_prices = []

for lr in forecast_log_ret_ar7:
    last_train_price = last_train_price * np.exp(lr)
    ar7_prices.append(last_train_price)

# Store in test dataframe
nifty_test['ar7_forecast'] = ar7_prices

In [ ]:
nifty_test.tail()

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(nifty_train['price'].iloc[-100:], color='steelblue', label='Train')
plt.plot(nifty_test['price'], color='green', label='Actual')
plt.title('AR(7) Model Forecast vs Actual NIFTY', fontweight='bold')
plt.plot(nifty_test['ar7_forecast'], color='red', 
         linestyle='--', label='AR(7) Forecast')
plt.plot(nifty_test['ar1_forecast'], color='blue', 
         linestyle='--', label='AR(1) Forecast')
plt.legend()
plt.show()

In [ ]:
# Checking the errors for both AR(1) and AR(7)
mae  = mean_absolute_error(nifty_test['price'], nifty_test['ar1_forecast'])
rmse = np.sqrt(mean_squared_error(nifty_test['price'], nifty_test['ar1_forecast']))
mape = np.mean(np.abs((nifty_test['price'] - nifty_test['ar1_forecast']) / nifty_test['price'])) * 100

print('=== AR(1) Model Errors ===')
print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAPE : {mape:.2f}%')

In [ ]:
mae  = mean_absolute_error(nifty_test['price'], nifty_test['ar7_forecast'])
rmse = np.sqrt(mean_squared_error(nifty_test['price'], nifty_test['ar7_forecast']))
mape = np.mean(np.abs((nifty_test['price'] - nifty_test['ar7_forecast']) / nifty_test['price'])) * 100

print('=== AR(7) Model Errors ===')
print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAPE : {mape:.2f}%')

Here we see that MAPE of AR(1) is less than the AR(7) model and other baseline models but not enough good predictor so now we have to proceed with MA model

**We are going to save ONLY AR(1) Model values since it was better than AR(7)**

In [ ]:
results.loc[3] = ['AR(1)', mae, rmse, mape]

### ***ALL THE DATA IN ONE GRAPH***

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(nifty_train['price'].iloc[-100:], 
         color='steelblue', label='Train', linewidth=1)
plt.plot(nifty_test['price'], 
         color='black', linewidth=2, label='Actual')
plt.plot(nifty_test['naive_forecast'], 
         '--', color='gray', linewidth=1, label='Naive')
plt.plot(nifty_test['avg_forecast'],   
         '--', color='purple', linewidth=1, label='Simple Avg')
plt.plot(nifty_test['30 days mean'],    
         '--', color='darkorange', linewidth=1, label='Moving Avg')
plt.plot(nifty_test['ar1_forecast'],    
         '-', color='red', linewidth=1.5, label='AR(1)')
plt.axvline(nifty_test.index[0], color='black', 
            linestyle=':', linewidth=1)
plt.title('All Models — Forecast vs Actual NIFTY', 
          fontsize=14, fontweight='bold')
plt.ylabel('Price (INR)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

## ***BAR GRAPH***

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    ax.bar(results['Model'], results[metric], 
           color=['steelblue','purple','darkorange','red','green'],
           edgecolor='white')
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('All Models — Error Metric Comparison', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## ***5.) MA(moving average) model***

"After fitting the AR model, we now apply the 
MA (Moving Average) model.

Unlike the AR model which uses past actual values, 
the MA model predicts future values using past 
forecast errors (residuals).

The equation is:

y(t) = μ + ε(t) + θ1*ε(t-1) + θ2*ε(t-2) + ... + θq*ε(t-q)

Where:
y(t)    = current value we want to predict
μ       = mean of the series
ε(t)    = current error
θ       = coefficients (weights given to past errors)
q       = number of past errors used (order)

The key parameter is q (order) which tells the 
model how many past errors to look at.

From our ACF plot we observed significant spikes 
at lag 5 and lag 6, which suggests q = 5 or 6.

Just like AR model, we will formally confirm 
the best order by:

1. Fitting MA models from MA(1) to MA(6)
2. Checking AIC and BIC at each step
3. Applying LLR test at each step

MA model is fitted on log returns (stationary data) 
and predictions are converted back to prices 
for final comparison."

In [ ]:
# We will plot ACF and see what lags are going to be significant
sgt.plot_acf(nifty_train.returns[1:] , lags=40)
plt.ylim(-.25,.25)
plt.show()

### "From the ACF plot we observe significant spikes at lag 3 and lag 6, which suggests that MA(3) or MA(6) could be the appropriate order for our MA model. However just like the AR model,we cannot rely on ACF plot alone. We will formally confirm the best order using AIC, BIC and LLR test."

In [ ]:
log_ret = nifty_train['returns'].dropna()

# MA(1)
model_ma1 = ARIMA(log_ret, order=(0,0,1))
ma1_fitted = model_ma1.fit()
print('=== MA(1) ===')
print(f'AIC : {ma1_fitted.aic:.4f}')
print(f'BIC : {ma1_fitted.bic:.4f}')
print(f'Log likelihood : {ma1_fitted.llf:.4f}')

In [ ]:
# MA(2)
model_ma2 = ARIMA(log_ret, order=(0,0,2))
ma2_fitted = model_ma2.fit()
print('=== MA(2) ===')
print(f'AIC : {ma2_fitted.aic:.4f}')
print(f'BIC : {ma2_fitted.bic:.4f}')
print(f'Log likelihood : {ma2_fitted.llf:.4f}')

In [ ]:
# LLR test MA(1) vs MA(2)
print(f'\nLLR Test MA(1) vs MA(2) : {LLR_test(model_ma1, model_ma2)}')

#### ***"The LLR test comparing MA(1) and MA(2) gives a p-value of 0.224 which is greater than 0.05. This means adding a second lag does not significantly improve the model. Though I would like to see the MA(3) and MA(6) also"***

In [ ]:
# MA(3)
model_ma3 = ARIMA(log_ret, order=(0,0,3))
ma3_fitted = model_ma3.fit()
print('=== MA(3) ===')
print(f'AIC : {ma3_fitted.aic:.4f}')
print(f'BIC : {ma3_fitted.bic:.4f}')
print(f'Log likelihood : {ma3_fitted.llf:.4f}')

# LLR test MA(1) vs MA(3)
print(f'\nLLR Test MA(1) vs MA(3) : {LLR_test(model_ma1, model_ma3 , DF=2)}')

`A very slight improvement in the model`

In [ ]:
# MA(6)
model_ma6 = ARIMA(log_ret, order=(0,0,6))
ma6_fitted = model_ma6.fit()
print('=== MA(6) ===')
print(f'AIC : {ma6_fitted.aic:.4f}')
print(f'BIC : {ma6_fitted.bic:.4f}')
print(f'Log likelihood : {ma6_fitted.llf:.4f}')

# LLR test MA(1) vs MA(6)
print(f'\nLLR Test MA(1) vs MA(6) : {LLR_test(model_ma1, model_ma6 , DF=5)}')

`Now Again like AR model` 
`(i). Log likelihood of MA(6) increased from 11194.79 to 11210.34`                                                          

`(ii). AIC went from -22383.5830 to -22404.6824 i.e it decreased which says that the model is a better fit `                                                    

`(iii). BIC went from -22364.7884 to -22354.5633 i.e increased it says that the model complexity increased`                

`(iv). LLR Test gave us p value = 0 which says fiiting of MA(6) is significant than MA(1)`                                    

`But we will conclude after Checking the MAPE Percentage`

In [ ]:
# Forecast log returns Using MA(1) Model
forecast_log_ret_ma1 = ma1_fitted.forecast(steps=len(nifty_test))
# Convert back to prices
last_train_price = nifty_train['price'].iloc[-1]
ma1_prices = []

for lr in forecast_log_ret_ma1:
    last_train_price = last_train_price * np.exp(lr)
    ma1_prices.append(last_train_price)
# Store in test dataframe
nifty_test['ma1_forecast'] = ma1_prices

In [ ]:
# Plot
plt.figure(figsize=(14,5))
plt.plot(nifty_train['price'], color='steelblue', label='Train')
plt.plot(nifty_test['price'], color='green', label='Actual')
plt.plot(nifty_test['ma1_forecast'], color='purple',
         linestyle='--', label='MA(1) Forecast')
plt.title('MA(1) Model Forecast vs Actual NIFTY', fontweight='bold')
plt.legend()
plt.show()

In [ ]:
# Forecast log returns Using MA(6) Model
forecast_log_ret_ma6 = ma6_fitted.forecast(steps=len(nifty_test))
# Convert back to prices
last_train_price = nifty_train['price'].iloc[-1]
ma6_prices = []

for lr in forecast_log_ret_ma6:
    last_train_price = last_train_price * np.exp(lr)
    ma6_prices.append(last_train_price)
# Store in test dataframe
nifty_test['ma6_forecast'] = ma6_prices

In [ ]:
# Plot
plt.figure(figsize=(14,5))
plt.plot(nifty_train['price'], color='steelblue', label='Train')
plt.plot(nifty_test['price'], color='green', label='Actual')
plt.plot(nifty_test['ma6_forecast'], color='purple',
         linestyle='--', label='MA(6) Forecast')
plt.title('MA(6) Model Forecast vs Actual NIFTY', fontweight='bold')
plt.legend()
plt.show()

In [ ]:
# Comparing the both plots
# Plot
plt.figure(figsize=(14,5))
plt.plot(nifty_train['price'][-100:], color='steelblue', label='Train')
plt.plot(nifty_test['price'], color='green', label='Actual')
plt.plot(nifty_test['ma1_forecast'], color='red',
         linestyle='--', label='MA(1) Forecast')
plt.plot(nifty_test['ma6_forecast'], color='purple',
         linestyle='--', label='MA(6) Forecast')
plt.title('MA(1) Model Forecast and MA(6) Model Forecast vs Actual NIFTY', fontweight='bold')
plt.legend()
plt.show()

`The differences are very minute`

In [ ]:
# Let us check the errors using residuals and compare 
# Errors MA(1) Model
mae  = mean_absolute_error(nifty_test['price'], nifty_test['ma1_forecast'])
rmse = np.sqrt(mean_squared_error(nifty_test['price'], nifty_test['ma1_forecast']))
mape = np.mean(np.abs((nifty_test['price'] - nifty_test['ma1_forecast']) / nifty_test['price'])) * 100

print('=== MA(1) Model Errors ===')
print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAPE : {mape:.2f}%')

In [ ]:
# Errors MA(6) Model
mae  = mean_absolute_error(nifty_test['price'], nifty_test['ma6_forecast'])
rmse = np.sqrt(mean_squared_error(nifty_test['price'], nifty_test['ma6_forecast']))
mape = np.mean(np.abs((nifty_test['price'] - nifty_test['ma6_forecast']) / nifty_test['price'])) * 100

print('=== MA(6) Model Errors ===')
print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAPE : {mape:.2f}%')

`So After comparing the errors b/w MA(1) and MA(6), we can say that MA(1) model actually forecast better than MA(6) even though it is a simple model`                                      

`Thus we will move forward with Only MA(1) Model`

In [ ]:
results.loc[4] = ['MA(1)', mae, rmse, mape]

Here the MAPE is less that the baseline models but not better than the AR model and also not the good predictor so we proceed with more advanced models

"We use a simple one time forecast approach 
where the model is trained once on the entire 
training data and then used to forecast all 
test period values at once. A more advanced 
approach would be walk forward validation 
where the model is retrained as each new data 
point becomes available, better simulating 
real world forecasting conditions."

## ***RESULT TABLE***

In [ ]:
# Print final results table
print('='*55)
print('        FINAL MODEL COMPARISON')
print('='*55)
print(results.to_string(index=False))
print()
print(f"Best model by MAPE : {results.loc[results['MAPE'].idxmin(), 'Model']}")

## ***BAR CHART COMPARISON***

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    ax.bar(results['Model'], results[metric], 
           color=['steelblue','purple','darkorange','red','green'],
           edgecolor='white')
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('All Models — Error Metric Comparison', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### ***All Forecasts on One Graph*** 

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(nifty_train['price'].iloc[-100:], 
         color='steelblue', label='Train', linewidth=1)
plt.plot(nifty_test['price'], 
         color='black', linewidth=2, label='Actual')
plt.plot(nifty_test['naive_forecast'], 
         '--', color='gray', linewidth=1, label='Naive')
plt.plot(nifty_test['avg_forecast'],   
         '--', color='purple', linewidth=1, label='Simple Avg')
plt.plot(nifty_test['30 days mean'],    
         '--', color='darkorange', linewidth=1, label='Moving Avg')
plt.plot(nifty_test['ar1_forecast'],    
         '-', color='red', linewidth=1.5, label='AR(1)')
plt.plot(nifty_test['ma1_forecast'],    
         '-', color='green', linewidth=1.5, label='MA(1)')

plt.axvline(nifty_test.index[0], color='black', 
            linestyle=':', linewidth=1)
plt.title('All Models — Forecast vs Actual NIFTY', 
          fontsize=14, fontweight='bold')
plt.ylabel('Price (INR)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

### ***ADVANCED MODELS*** 

### ***(i). ARMA Model*** 

In [ ]:
# We will start with ARMA(1,1) i.e ARIMA(1,0,1) on log returns
model_arma_11 = ARIMA(log_ret , order = (1,0,1))
result_arma_11 = model_arma_11.fit()
print(result_arma_11.summary())
print('\n')
#-------- Residuals Findings -------------

resid_arma_11 = result_arma_11.resid[1:]

In [ ]:
plt.figure(figsize=(20,5))
plt.plot(resid_arma_11[1:],color='g',label='Train Residuals ARMA(1,1)')
plt.axhline(nifty_df.returns.mean(),color='k',linestyle='--',label='mean line')
plt.title("Training Residuals on Log Returns NIFTY50",size=24)
plt.legend()
plt.show()

In [ ]:
# Checking the Stationarity of Residual using Augmented Dickey Fuller Test
sts.adfuller(resid_arma_11[1:])

**p value <0.05 implies that The Residuals are Stationary**

In [ ]:
# Plot ACF On Residuals
sgt.plot_acf(resid_arma_11[1:],zero=False,lags=40)
plt.ylim(-.07,.07)
plt.title("ACF ON RESIDUALS OF ARIMA(1,0,1) Model",size=24)
plt.show()

In [ ]:
# Plot PACF On Residuals
sgt.plot_pacf(resid_arma_11[1:],zero=False,lags=40 , method='ols')
plt.ylim(-.07,.07)
plt.title("PACF ON RESIDUALS OF ARIMA(1,0,1) Model",size=24)
plt.show()

`In ACF plot, There are significant lags at lag = 3, 6 -> So we should Try using MA component = 3 or 6`                                                    
`In PACF plot, similarly there are significant lags at lag = 3, 6 -> So we should Try using AR component = 3 or 6`                                                    

In [ ]:
# Let us fit Ljung Box Test to see if Autocorrelation still exists in the Residuals
from statsmodels.stats.diagnostic import acorr_ljungbox

acorr_ljungbox(resid_arma_11, lags=[10], return_df=True)


**We see that p value < 0.05 implies that The Residuals still have Autocorrelation. This means that we should Try higher p or q model**

`Let us find the Forecast and tell plot the data`

In [ ]:
# Forecast log returns Using ARMA(1,1) Model
forecast_ret_arma_11 = result_arma_11.forecast(steps=len(nifty_test))
# Convert back to prices
last_train_price = nifty_train['price'].iloc[-1]
arma_11_prices = []

for lr in forecast_ret_arma_11:
    last_train_price = last_train_price * np.exp(lr)
    arma_11_prices.append(last_train_price)
# Store in test dataframe
nifty_test['arma_11_forecast'] = arma_11_prices

In [ ]:
# Plot
plt.figure(figsize=(14,5))
plt.plot(nifty_train['price'], color='steelblue', label='Train')
plt.plot(nifty_test['price'], color='green', label='Actual')
plt.plot(nifty_test['arma_11_forecast'], color='purple',
         linestyle='--', label='ARMA(1,1) Forecast')
plt.title('ARMA(1,1) Model Forecast vs Actual NIFTY', fontweight='bold')
plt.legend()
plt.show()

In [ ]:
# Plotting Forecasting Residuals of ARMA(1,1) Model

plt.figure(figsize=(12,6))

# residual line
plt.plot(nifty_test['price'] - nifty_test['arma_11_forecast'], color='red', label='Forecast Residuals')

# zero reference line
plt.axhline(y=0, color='black', linestyle='--')

# titles
plt.title("Residuals (Actual – Forecast)")

# labels
plt.xlabel("Date")
plt.ylabel("Residuals")

plt.legend()
plt.show()

**Forecast Residuals show several serious problems.**

From around 2024 to 2026:

residuals remain strongly positive for long periods

Meaning:

Actual>Forecast

So model is consistently underpredicting.

This indicates:

systematic bias
model missing trend/structure

`A Well fitted Residual model looks like a Random Noise`

In [ ]:
# Errors ARMA(1,1) Model
mae  = mean_absolute_error(nifty_test['price'], nifty_test['arma_11_forecast'])
rmse = np.sqrt(mean_squared_error(nifty_test['price'], nifty_test['arma_11_forecast']))
mape = np.mean(np.abs((nifty_test['price'] - nifty_test['arma_11_forecast']) / nifty_test['price'])) * 100

print('=== ARMA(1,1) Model Errors ===')
print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAPE : {mape:.2f}%')

In [ ]:
results.loc[5] = ['ARMA(1,1)', mae, rmse, mape]

**Fitting ARMA(1,3) , ARMA(3,1) , ARMA(1,6) , ARMA(6,1) , ARMA(3,3) , ARMA(3,6) , ARMA(6,1) , ARMA(6,6)**

In [ ]:
for i in [1,3,6]:
    for j in [1,3,6]:
        if (i,j) != (1,1):
            models = ARIMA(log_ret , order = (i,0,j))
            result = models.fit()
            print(result.summary())
            print('\n')

**CONCLUSION :**
***We can clearly see that ARMA(6,6) model performs the best among all the models***

#### Let us quickly check the LLR Test b/w ARMA(1,1) and ARMA(6,6) To see if Complex model is better or not

In [ ]:
# We will start with ARMA(6,6) i.e ARIMA(6,0,6) on log returns
model_arma_66 = ARIMA(log_ret , order = (6,0,6))
result_arma_66 = model_arma_66.fit()
print(result_arma_66.summary())
print('\n')
#-------- Residuals Findings -------------

resid_arma_66 = result_arma_66.resid[1:]

In [ ]:
# Plotting the Training Residuals
plt.figure(figsize=(20,5))
plt.plot(resid_arma_66[1:],color='g',label='Train Residuals ARMA(6,6)')
plt.axhline(nifty_df.returns.mean(),color='k',linestyle='--',label='mean line')
plt.title("Training Residuals on Log Returns NIFTY50",size=24)
plt.legend()
plt.show()

In [ ]:
# Checking the Stationarity of Residual using Augmented Dickey Fuller Test
sts.adfuller(resid_arma_66[1:])

`This Test shows that the Residuals of ARMA(6,6) are Highly Stationary`

In [ ]:
# Plot ACF On Residuals
sgt.plot_acf(resid_arma_66,zero=False,lags=40)
plt.ylim(-.07,.07)
plt.title("ACF ON RESIDUALS OF ARIMA(6,0,6) Model",size=24)
plt.show()

In [ ]:
# Plot PACF On Residuals
sgt.plot_pacf(resid_arma_66,zero=False,lags=40 , method='ols')
plt.ylim(-.07,.07)
plt.title("PACF ON RESIDUALS OF ARIMA(6,0,6) Model",size=24)
plt.show()

In [ ]:
# Let us fit Ljung Box Test to see if Autocorrelation still exists in the Residuals
from statsmodels.stats.diagnostic import acorr_ljungbox

acorr_ljungbox(resid_arma_66, lags=[10], return_df=True)

`See the Ljung Box Test, we got p value >0.05 which implies our Residual does not have Autocorrelation left`                                       
`Thus we can consider our Residual to be White Noise`

**Let us find the forecast values and then plot the Data with forecast and Also Forecast Residuals**

In [ ]:
# Forecast log returns Using ARMA(6,6) Model
forecast_ret_arma_66 = result_arma_66.forecast(steps=len(nifty_test))
# Convert back to prices
last_train_price = nifty_train['price'].iloc[-1]
arma_66_prices = []

for lr in forecast_ret_arma_66:
    last_train_price = last_train_price * np.exp(lr)
    arma_66_prices.append(last_train_price)
# Store in test dataframe
nifty_test['arma_66_forecast'] = arma_66_prices

# --------Plot-------------------
plt.figure(figsize=(14,5))
plt.plot(nifty_train['price'], color='steelblue', label='Train')
plt.plot(nifty_test['price'], color='green', label='Actual')
plt.plot(nifty_test['arma_66_forecast'], color='purple',
         linestyle='--', label='ARMA(6,6) Forecast')
plt.title('ARMA(6,6) Model Forecast vs Actual NIFTY', fontweight='bold')
plt.legend()
plt.show()

In [ ]:
# Plotting Forecasting Residuals of ARMA(6,6) Model

plt.figure(figsize=(12,6))

# residual line
plt.plot(nifty_test['price'] - nifty_test['arma_66_forecast'], color='red', label='Forecast Residuals')

# zero reference line
plt.axhline(y=0, color='black', linestyle='--')

# titles
plt.title("Residuals (Actual – Forecast)")

# labels
plt.xlabel("Date")
plt.ylabel("Residuals")

plt.legend()
plt.show()

In [ ]:
# Errors ARMA(1,1) Model
mae  = mean_absolute_error(nifty_test['price'], nifty_test['arma_66_forecast'])
rmse = np.sqrt(mean_squared_error(nifty_test['price'], nifty_test['arma_66_forecast']))
mape = np.mean(np.abs((nifty_test['price'] - nifty_test['arma_66_forecast']) / nifty_test['price'])) * 100

print('=== ARMA(6,6) Model Errors ===')
print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAPE : {mape:.2f}%')

In [ ]:
results.loc[6] = ['ARMA(6,6)', mae, rmse, mape]

## ***Result Table***

In [ ]:
# Print final results table
print('='*55)
print('        FINAL MODEL COMPARISON')
print('='*55)
print(results.to_string(index=False))
print()
print(f"Best model by MAPE : {results.loc[results['MAPE'].idxmin(), 'Model']}")

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    ax.bar(results['Model'], results[metric], 
           color=['steelblue','purple','darkorange','red','green'],
           edgecolor='white')
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('All Models — Error Metric Comparison', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## ***So After All these fitting We can Say that ARMA(6,6) is the best fit model so far***

### ***(ii). ARIMA MODEL*** 

In [ ]:
model_arima_111 = ARIMA(nifty_train['price'] , order=(1,1,1))
result_arima_111 = model_arima_111.fit()
print(result_arima_111.summary())

#--------------- Training Residual --------------------

resid_arima_111 = result_arima_111.resid[1:]

**We use d = 1 for differencing once since we know that On differencing once our prices changes into Stationary**

In [ ]:
# Checking the Stationarity of Residual using Augmented Dickey Fuller Test
sts.adfuller(resid_arima_111[1:])

In [ ]:
# Plot ACF On Residuals
sgt.plot_acf(resid_arima_111,zero=False,lags=40)
plt.ylim(-.07,.07)
plt.title("ACF ON RESIDUALS OF ARIMA(1,1,1) Model",size=24)
plt.show()

In [ ]:
# Plot PACF On Residuals
sgt.plot_pacf(resid_arima_111,zero=False,lags=40 , method='ols')
plt.ylim(-.07,.07)
plt.title("PACF ON RESIDUALS OF ARIMA(1,1,1) Model",size=24)
plt.show()

`Lags = 5 and Lag = 6 looks highly significant, we should check ARIMA with p =5/6 or q = 5/6`

`This shows that The Residuals are Stationary`

In [ ]:
# Let us fit Ljung Box Test to see if Autocorrelation still exists in the Residuals
acorr_ljungbox(resid_arima_111, lags=[10], return_df=True)

`There exist Autocorrelation between the Lags since p value <0.05`

In [ ]:
# Plotting the Training Residuals
plt.figure(figsize=(20,5))
plt.plot(resid_arima_111[1:],color='g',label='Train Residuals ARIMA(1,1,1)')
plt.axhline(nifty_df.returns.mean(),color='k',linestyle='--',label='mean line')
plt.title("Training Residuals on Log Returns NIFTY50",size=24)
plt.legend()
plt.show()

In [ ]:
# Forecast log returns Using ARIMA(1,1,1) Model
forecast_arima_111 = result_arima_111.forecast(steps=len(nifty_test))

# Store in test dataframe
nifty_test['arima_111_forecast'] = forecast_arima_111

# --------Plot-------------------
plt.figure(figsize=(14,5))
plt.plot(nifty_train['price'], color='steelblue', label='Train')
plt.plot(nifty_test['price'], color='green', label='Actual')
plt.plot(nifty_test['arma_66_forecast'], color='purple',
         linestyle='--', label='ARIMA(1,1,1) Forecast')
plt.title('ARIMA(1,1,1) Model Forecast vs Actual NIFTY', fontweight='bold')
plt.legend()
plt.show()

In [ ]:
# Plotting Forecasting Residuals of ARMA(6,6) Model

plt.figure(figsize=(12,6))

# residual line
plt.plot(nifty_test['price'] - nifty_test['arima_111_forecast'], color='red', label='Forecast Residuals')

# zero reference line
plt.axhline(y=0, color='black', linestyle='--')

# titles
plt.title("Residuals (Actual – Forecast)")

# labels
plt.xlabel("Date")
plt.ylabel("Residuals")

plt.legend()
plt.show()

In [ ]:
# Errors ARMA(1,1) Model
mae  = mean_absolute_error(nifty_test['price'], nifty_test['arima_111_forecast'])
rmse = np.sqrt(mean_squared_error(nifty_test['price'], nifty_test['arima_111_forecast']))
mape = np.mean(np.abs((nifty_test['price'] - nifty_test['arima_111_forecast']) / nifty_test['price'])) * 100

print('=== ARIMA(1,1,1) Model Errors ===')
print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'MAPE : {mape:.2f}%')

`We see higher errors in fitting ARIMA(1,1,1) on prices than fitting ARMA(6,6) on log returns`

**Most of the time instead of Fitting ARIMA model on prices we try to fit in log_price instead**

In [ ]:
# So we will try to fit on log_price and then check ARIMA(1,1,1) and compare the results

nifty_train['log_price'] = np.log(nifty_train['price'])
nifty_test['log_price'] = np.log(nifty_test['price'])

In [ ]:
model_arima_111_logprice = ARIMA(nifty_train['log_price'] , order=(1,1,1))
result_arima_111_logprice = model_arima_111_logprice.fit()
print(result_arima_111_logprice.summary())

#--------------- Training Residual --------------------

resid_arima_111_logprice = result_arima_111_logprice.resid[1:]

In [ ]:
# Forecast log prices Using ARIMA(1,1,1) Model and then change them back
forecast_arima_111_logprice = result_arima_111_logprice.forecast(steps=len(nifty_test))

# change back to price 
forecast_arima_111_price = np.exp(forecast_arima_111_logprice)
# Store in test dataframe
nifty_test['arima_111_forecast_price'] = forecast_arima_111_price

# --------Plot-------------------
plt.figure(figsize=(14,5))
plt.plot(nifty_train['price'], color='steelblue', label='Train')
plt.plot(nifty_test['price'], color='green', label='Actual')
plt.plot(nifty_test['arima_111_forecast_price'], color='purple',
         linestyle='--', label='ARIMA(1,1,1) Forecast On Pric')
plt.title('ARIMA(1,1,1) Model Forecast After Changing back to Price From Log Price vs Actual NIFTY', fontweight='bold')
plt.legend()
plt.show()

### ***Why Raw Price ARIMA Looks Better Here — But Is Actually Fragile***
This is the nuanced part. Raw price ARIMA looks better in the plot, but for the wrong reason.
Raw price ARIMA is essentially learning: "prices go up by a large absolute number each day" — because NIFTY went from 5000 to 17500 over the training period and the AR term captures that absolute drift. When it forecasts forward, it continues that same absolute per-day increment, which happens to look like it's tracking the actual upward movement.
But this is brittle:

If the market had gone sideways for 2 years then crashed, the raw ARIMA would still project upward
The forecast has no awareness of percentage-based movement, only absolute
It's essentially extrapolating a linear trend, not modeling the process

Log price ARIMA is theoretically more correct for financial data. It just happens that when you forecast 500+ steps with near-zero drift, it looks flat — because ARIMA is a linear model and cannot capture the exponential compounding nature of equity markets over long horizons.